# Hypothesis Testing

## Learning Objectives

By the end of this notebook you will be able to:

- Understand what a hypothesis test is and why we use it
- Correctly interpret the **significance level** ($\alpha$) and **statistical power** in business contexts
- Interpret the **p-value** correctly
- Recognize **Type I** and **Type II** errors
- Calculate an appropriate **sample size** for a study
- Connect statistical results to **real-world conclusions**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.stats as stats
from scipy.stats import ttest_1samp, ttest_ind, norm

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

## 1. What Is Hypothesis Testing?

A **hypothesis test** is a formal procedure to decide whether evidence from a sample is strong enough to reject a claim about a population.

### Key components

| Symbol | Name | Meaning |
|--------|------|---------|
| $H_0$ | Null hypothesis | The default assumption (status quo). We assume it is **true** unless the data says otherwise. |
| $H_1$ (or $H_a$) | Alternative hypothesis | What we want to prove. It contradicts $H_0$. |
| $\alpha$ | Significance level | Maximum tolerated probability of a false alarm. |
| Power | $1 - \beta$ | Probability of detecting a real effect when it exists. |
| p-value | — | Probability of observing the data (or more extreme) assuming $H_0$ is true. |

### Decision rule

$$
\text{If } p\text{-value} < \alpha \Rightarrow \text{Reject } H_0 \quad \text{(statistically significant)}
$$
$$
\text{If } p\text{-value} \geq \alpha \Rightarrow \text{Fail to reject } H_0 \quad \text{(not significant)}
$$

> **Important:** "Fail to reject $H_0$" is **not** the same as "$H_0$ is true".

---

## 2. Significance Level ($\alpha$)

### What is it?

The **significance level** $\alpha$ is the maximum probability we are willing to tolerate of making a **false alarm** — rejecting $H_0$ when it is actually true (a Type I error).

$$\alpha = P(\text{Reject } H_0 \mid H_0 \text{ is true})$$

We set $\alpha$ **before** collecting any data. Common choices:

| $\alpha$ | Meaning | Typical context |
|----------|---------|----------------|
| 0.10 | 10% false alarm rate | Exploratory research, low-stakes decisions |
| **0.05** | **5% false alarm rate** | **Standard in most business and academic work** |
| 0.01 | 1% false alarm rate | High-stakes decisions (medical, financial regulation) |
| 0.001 | 0.1% false alarm rate | Safety-critical systems, drug approvals |

### The critical region

Choosing $\alpha$ divides the distribution of the test statistic into:
- **Non-rejection region:** values we would expect if $H_0$ is true.
- **Rejection region (critical region):** extreme values that lead us to reject $H_0$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.linspace(-4, 4, 500)
y = norm.pdf(x)

alphas     = [0.10, 0.05, 0.01]
titles     = ['α = 0.10\n(Lenient — more false alarms)',
               'α = 0.05\n(Standard)',
               'α = 0.01\n(Strict — fewer false alarms)']
crit_colors = ['darkorange', 'tomato', 'darkred']

for ax, alpha_val, title, color in zip(axes, alphas, titles, crit_colors):
    crit = norm.ppf(1 - alpha_val / 2)  # two-tailed
    ax.plot(x, y, 'steelblue', linewidth=2)
    ax.fill_between(x, y, where=(x >= crit),  color=color, alpha=0.55, label=f'Rejection region')
    ax.fill_between(x, y, where=(x <= -crit), color=color, alpha=0.55)
    ax.fill_between(x, y, where=((x > -crit) & (x < crit)), color='lightblue', alpha=0.3, label='Non-rejection region')
    ax.axvline( crit, color=color, linestyle='--', linewidth=1.8, label=f'±{crit:.2f}')
    ax.axvline(-crit, color=color, linestyle='--', linewidth=1.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Test statistic (z)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 0.45)

plt.suptitle('How the significance level α controls the size of the rejection region', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Business Example 1 — Choosing α for an Email Campaign

A marketing team tests whether a **new email subject line** increases the open rate above the current 20%.

- $H_0$: open rate = 20% (new subject is no better)
- $H_1$: open rate > 20% (new subject is better)

**Why does α matter here?**

| α choice | Business consequence of a Type I Error |
|----------|----------------------------------------|
| α = 0.10 | 10% chance of rolling out a subject line that is actually no better — wasted design effort |
| α = 0.05 | 5% chance — the standard trade-off |
| α = 0.01 | 1% chance — very conservative, but we might miss a real improvement |

For a low-cost email campaign, **α = 0.05** is reasonable. For a campaign requiring expensive creative production, **α = 0.01** is safer.

In [ ]:
# Simulate email campaign data
# Current open rate: 20%. We send the new subject to n=500 subscribers.
np.random.seed(7)
n_emails = 500
p0 = 0.20   # claimed baseline rate

# Scenario A: new subject is equally good (true rate = 20%)
opens_A = np.random.binomial(n=1, p=0.20, size=n_emails)
# Scenario B: new subject is genuinely better (true rate = 24%)
opens_B = np.random.binomial(n=1, p=0.24, size=n_emails)

def z_test_proportion(successes, n, p0, alternative='greater'):
    """One-sample proportion z-test."""
    p_hat = successes / n
    se = np.sqrt(p0 * (1 - p0) / n)
    z = (p_hat - p0) / se
    if alternative == 'greater':
        p_val = 1 - norm.cdf(z)
    elif alternative == 'less':
        p_val = norm.cdf(z)
    else:
        p_val = 2 * (1 - norm.cdf(abs(z)))
    return p_hat, z, p_val

for label, opens in [('Scenario A (true rate = 20%)', opens_A),
                      ('Scenario B (true rate = 24%)', opens_B)]:
    p_hat, z, p_val = z_test_proportion(opens.sum(), n_emails, p0)
    print(f"--- {label} ---")
    print(f"  Observed open rate : {p_hat:.1%}")
    print(f"  z-statistic        : {z:.3f}")
    print(f"  p-value            : {p_val:.4f}")
    for alpha_val in [0.10, 0.05, 0.01]:
        decision = 'REJECT H₀ ✓' if p_val < alpha_val else 'fail to reject'
        print(f"  α = {alpha_val} → {decision}")
    print()

### Visualizing the false alarm rate over many experiments

In [ ]:
# Simulate 2000 experiments where H0 is TRUE (no real effect)
# and count how often we wrongly reject H0 at different α levels
np.random.seed(0)
n_experiments = 2000
n_per_exp = 200
p_true = 0.20  # H0 is true

p_values_null = []
for _ in range(n_experiments):
    sample = np.random.binomial(1, p_true, n_per_exp)
    _, _, p_val = z_test_proportion(sample.sum(), n_per_exp, p_true)
    p_values_null.append(p_val)

p_values_null = np.array(p_values_null)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of p-values under H0 (should be uniform)
axes[0].hist(p_values_null, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
for alpha_val, color in [(0.10, 'darkorange'), (0.05, 'tomato'), (0.01, 'darkred')]:
    axes[0].axvline(alpha_val, color=color, linestyle='--', linewidth=1.8, label=f'α = {alpha_val}')
axes[0].set_xlabel('p-value', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of p-values when H₀ is TRUE\n(2000 simulated experiments)', fontsize=12)
axes[0].legend()

# Right: false alarm rate at each alpha
alpha_grid = np.linspace(0, 0.20, 100)
false_alarm_rates = [(p_values_null < a).mean() for a in alpha_grid]
axes[1].plot(alpha_grid, false_alarm_rates, 'steelblue', linewidth=2, label='Observed false alarm rate')
axes[1].plot(alpha_grid, alpha_grid, 'k--', linewidth=1.5, label='Expected (= α)')
for alpha_val, color in [(0.10, 'darkorange'), (0.05, 'tomato'), (0.01, 'darkred')]:
    rate = (p_values_null < alpha_val).mean()
    axes[1].scatter([alpha_val], [rate], color=color, zorder=5, s=80,
                    label=f'α={alpha_val}: {rate:.1%} false alarms')
axes[1].set_xlabel('Significance level α', fontsize=12)
axes[1].set_ylabel('False alarm rate', fontsize=12)
axes[1].set_title('α directly controls the false alarm rate', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight: when H₀ is true, p-values are uniformly distributed.")
print("α = 0.05 means ~5% of true-null experiments will be wrongly flagged as significant.")

---

## 3. Statistical Power ($1 - \beta$)

### What is it?

**Power** is the probability of correctly detecting a real effect when it truly exists.

$$\text{Power} = 1 - \beta = P(\text{Reject } H_0 \mid H_1 \text{ is true})$$

where $\beta = P(\text{Type II Error}) = P(\text{Fail to reject } H_0 \mid H_1 \text{ is true})$.

### Power depends on four things

| Factor | Effect on Power |
|--------|-----------------|
| **Effect size** ↑ | Power ↑ — bigger effects are easier to detect |
| **Sample size** ↑ | Power ↑ — more data → less uncertainty |
| **α** ↑ | Power ↑ — but at cost of more false alarms |
| **σ (noise)** ↑ | Power ↓ — more noise makes signals harder to see |

### Why does power matter in business?

Low power means you run an experiment, see "no significant effect", and wrongly conclude the campaign/feature/product is not working — **when it actually is**. This is a missed opportunity.

> Industry standard: aim for **power ≥ 80%** (often 90% for high-stakes decisions).

In [ ]:
# --- Visual: H0 vs H1, showing alpha and power regions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Three scenarios: same alpha, different effect sizes
scenarios = [
    {'delta': 1.0, 'label': 'Small effect (d = 1.0)'},
    {'delta': 2.0, 'label': 'Medium effect (d = 2.0)'},
    {'delta': 3.0, 'label': 'Large effect (d = 3.0)'},
]
alpha_val = 0.05
critical = norm.ppf(1 - alpha_val)  # one-tailed
x = np.linspace(-4, 8, 600)

for ax, sc in zip(axes, scenarios):
    delta = sc['delta']
    y0 = norm.pdf(x, loc=0, scale=1)
    y1 = norm.pdf(x, loc=delta, scale=1)
    power_val = 1 - norm.cdf(critical, loc=delta, scale=1)
    beta_val  = norm.cdf(critical, loc=delta, scale=1)

    ax.plot(x, y0, 'steelblue', linewidth=2, label='H₀ (no effect)')
    ax.plot(x, y1, 'seagreen',  linewidth=2, label='H₁ (real effect)')
    ax.fill_between(x, y0, where=(x >= critical), color='tomato',  alpha=0.5, label=f'α = {alpha_val}')
    ax.fill_between(x, y1, where=(x >= critical), color='seagreen',alpha=0.5, label=f'Power = {power_val:.0%}')
    ax.fill_between(x, y1, where=(x <  critical), color='gray',    alpha=0.3, label=f'β = {beta_val:.0%}')
    ax.axvline(critical, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(sc['label'], fontsize=11)
    ax.set_xlabel('Test statistic', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=7.5, loc='upper right')
    ax.set_xlim(-4, 8)

plt.suptitle('Power increases as the true effect size grows (α = 0.05, one-tailed)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Power also increases with sample size

In [ ]:
# Power vs sample size for a fixed effect (Cohen's d = 0.3 — small/medium)
# One-sample z-test, alpha=0.05, two-tailed
from scipy.stats import norm as normal

def compute_power(n, effect_size, alpha=0.05):
    """Power for a one-sample z-test."""
    z_alpha = normal.ppf(1 - alpha / 2)
    z_beta  = effect_size * np.sqrt(n) - z_alpha
    return normal.cdf(z_beta)

n_range = np.arange(5, 500, 5)
effect_sizes = [0.2, 0.3, 0.5, 0.8]
colors_p = ['tomato', 'darkorange', 'steelblue', 'seagreen']
labels_p = ['Small (d=0.2)', 'Small/med (d=0.3)', 'Medium (d=0.5)', 'Large (d=0.8)']

fig, ax = plt.subplots(figsize=(11, 6))
for d, color, label in zip(effect_sizes, colors_p, labels_p):
    powers = [compute_power(n, d) for n in n_range]
    ax.plot(n_range, powers, color=color, linewidth=2, label=label)

ax.axhline(0.80, color='black', linestyle=':', linewidth=1.5, label='80% power target')
ax.axhline(0.90, color='gray',  linestyle=':', linewidth=1.5, label='90% power target')
ax.set_xlabel('Sample size (n)', fontsize=12)
ax.set_ylabel('Power (1 − β)', fontsize=12)
ax.set_title('Statistical power vs. sample size for different effect sizes\n(α = 0.05, two-tailed, one-sample test)', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

# Minimum n to reach 80% power
print("Minimum n for 80% power (α=0.05, one-sample, two-tailed):")
for d, label in zip(effect_sizes, labels_p):
    for n in range(5, 2000):
        if compute_power(n, d) >= 0.80:
            print(f"  {label:<22}: n = {n}")
            break

### Business Example 2 — Underpowered A/B Test in Marketing

A digital marketing agency tests a **new landing page** against the current one.

- Current conversion rate: 5%
- The new page is expected to increase conversions to 6% (a 1-percentage-point lift)
- The team runs the test for just **1 week**, collecting only 400 users per variant

**What is the power of this test?** Let's find out — and simulate the consequence of low power.

In [ ]:
np.random.seed(21)

p_control  = 0.05   # current landing page
p_treatment = 0.06  # new landing page (true effect exists)
n_per_group = 400

# Effect size for proportions (Cohen's h)
cohens_h = 2 * np.arcsin(np.sqrt(p_treatment)) - 2 * np.arcsin(np.sqrt(p_control))
print(f"Cohen's h (effect size for proportions) : {cohens_h:.4f}")

# Analytical power
alpha = 0.05
z_alpha = norm.ppf(1 - alpha / 2)
# Pooled SE under H0
p_pool = (p_control + p_treatment) / 2
se     = np.sqrt(2 * p_pool * (1 - p_pool) / n_per_group)
z_power = (p_treatment - p_control) / se - z_alpha
power_analytical = norm.cdf(z_power)
print(f"Analytical power with n={n_per_group}/group : {power_analytical:.1%}")
print()

# Simulate 1000 replications of this underpowered experiment
n_sims = 1000
detected = 0
for _ in range(n_sims):
    ctrl  = np.random.binomial(1, p_control,   n_per_group)
    treat = np.random.binomial(1, p_treatment, n_per_group)
    # Two-proportion z-test
    p1, p2 = ctrl.mean(), treat.mean()
    p_pool_sim = (ctrl.sum() + treat.sum()) / (2 * n_per_group)
    se_sim = np.sqrt(p_pool_sim * (1 - p_pool_sim) * (1/n_per_group + 1/n_per_group))
    if se_sim == 0:
        continue
    z = (p2 - p1) / se_sim
    p_val = 1 - norm.cdf(z)  # one-tailed: treatment > control
    if p_val < alpha:
        detected += 1

simulated_power = detected / n_sims
print(f"Simulated power ({n_sims} experiments) : {simulated_power:.1%}")
print()
print(f"→ In {n_sims} replications, the team correctly detects the 1pp lift only ~{simulated_power:.0%} of the time.")
print(f"→ ~{1-simulated_power:.0%} of the time they conclude 'no effect' — and potentially scrap a page that actually works.")

In [ ]:
# Show power for different sample sizes
n_options = [200, 400, 800, 1500, 3000]
powers_marketing = []

for n in n_options:
    se_n = np.sqrt(2 * p_pool * (1 - p_pool) / n)
    z_pw = (p_treatment - p_control) / se_n - z_alpha
    powers_marketing.append(norm.cdf(z_pw))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([str(n) for n in n_options], powers_marketing,
              color=['tomato' if p < 0.80 else 'seagreen' for p in powers_marketing],
              edgecolor='white', linewidth=1.2)

ax.axhline(0.80, color='black', linestyle='--', linewidth=1.8, label='80% power target')
for bar, pwr in zip(bars, powers_marketing):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{pwr:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Sample size per group (users)', fontsize=12)
ax.set_ylabel('Statistical Power', fontsize=12)
ax.set_title('Power of the landing page A/B test vs. sample size\n'
             '(detecting a 5% → 6% conversion lift, α=0.05)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)

red_patch   = mpatches.Patch(color='tomato',   label='Underpowered (< 80%)')
green_patch = mpatches.Patch(color='seagreen', label='Adequately powered (≥ 80%)')
ax.legend(handles=[red_patch, green_patch, plt.Line2D([0],[0],color='black',linestyle='--',label='80% target')],
          fontsize=10)

plt.tight_layout()
plt.show()

### Key takeaway

The 1 percentage-point lift (5% → 6%) is a **small effect**. With only 400 users per group, the test is severely **underpowered** — we miss the real improvement most of the time. The business needs ~1,500–3,000 users per group to reliably detect this kind of lift.

---

## 4. The α–Power Trade-off

For a **fixed sample size** and effect, there is a direct tension between $\alpha$ and power:

- Decreasing $\alpha$ (stricter) → fewer false positives, but **lower power** (more misses)
- Increasing $\alpha$ (lenient) → more power, but **more false alarms**

The only way to have **both** low $\alpha$ and high power is to **increase the sample size**.

### Business Example 3 — Newsletter Subscription Rate

A media company tests whether a **pop-up discount offer** increases newsletter sign-ups.

- Current sign-up rate: 8%
- Expected new rate: 10%
- Fixed budget: 600 visitors total (300 per group)

The team must decide: use α = 0.05 or α = 0.10?

In [ ]:
p_ctrl  = 0.08
p_treat = 0.10
n_group = 300
p_pool_trade = (p_ctrl + p_treat) / 2
se_trade = np.sqrt(2 * p_pool_trade * (1 - p_pool_trade) / n_group)

alpha_range = np.linspace(0.005, 0.20, 200)
powers_tradeoff = []
for a in alpha_range:
    z_a = norm.ppf(1 - a)           # one-tailed
    z_b = (p_treat - p_ctrl) / se_trade - z_a
    powers_tradeoff.append(norm.cdf(z_b))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trade-off curve
axes[0].plot(alpha_range, powers_tradeoff, 'steelblue', linewidth=2.5)
for a_mark, color in [(0.05, 'tomato'), (0.10, 'darkorange')]:
    z_a = norm.ppf(1 - a_mark)
    pwr = norm.cdf((p_treat - p_ctrl) / se_trade - z_a)
    axes[0].scatter([a_mark], [pwr], color=color, s=100, zorder=5,
                    label=f'α={a_mark}: power={pwr:.0%}')
axes[0].axhline(0.80, color='gray', linestyle=':', linewidth=1.5)
axes[0].set_xlabel('Significance level (α)', fontsize=12)
axes[0].set_ylabel('Power (1 − β)', fontsize=12)
axes[0].set_title('α–Power trade-off\n(n=300/group, 8%→10% lift)', fontsize=12)
axes[0].legend(fontsize=10)

# Right: stacked bar showing α and β for each choice
choices   = ['α = 0.05', 'α = 0.10']
alphas_ch = [0.05, 0.10]
betas_ch  = []
for a in alphas_ch:
    z_a = norm.ppf(1 - a)
    pwr = norm.cdf((p_treat - p_ctrl) / se_trade - z_a)
    betas_ch.append(1 - pwr)

x_pos = np.arange(len(choices))
w = 0.35
bars1 = axes[1].bar(x_pos - w/2, alphas_ch, w, label='α (false alarm risk)', color='tomato', alpha=0.85)
bars2 = axes[1].bar(x_pos + w/2, betas_ch,  w, label='β (miss risk)',        color='steelblue', alpha=0.85)
for bar, val in zip(list(bars1) + list(bars2), alphas_ch + betas_ch):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.0%}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(choices, fontsize=12)
axes[1].set_ylabel('Error probability', fontsize=12)
axes[1].set_title('Error rates for each α choice\n(n=300/group)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 0.75)

plt.tight_layout()
plt.show()

print("With n=300/group:")
for a in [0.05, 0.10]:
    z_a = norm.ppf(1 - a)
    pwr = norm.cdf((p_treat - p_ctrl) / se_trade - z_a)
    print(f"  α={a:.2f} → Power={pwr:.1%}, β={1-pwr:.1%}")

### Real-world interpretation

With 300 users per group:

- **α = 0.05:** Only ~33% chance of detecting the real lift — the test is dangerously underpowered. Most of the time the company will wrongly abandon the pop-up.
- **α = 0.10:** Power improves to ~45% — better, but still underwhelming. We now accept a 10% false alarm rate.

The real solution is **more data**, not a looser threshold. Relaxing $\alpha$ to compensate for small samples is a form of *p-hacking* that inflates false discoveries.

> **Rule of thumb:** Set $\alpha$ based on business risk tolerance *first*, then determine the sample size needed to achieve the desired power at that $\alpha$.

---

### Business Example 4 — A/B Test Comparison Across Three Campaigns

A company runs three separate marketing campaigns simultaneously. Each campaign tests a different channel against a control.

| Campaign | Metric | Control | Treatment | Sample size (each) |
|----------|--------|---------|-----------|--------------------|
| Email    | Open rate | 20% | 23% | 1,000 |
| Social   | Click rate | 3% | 3.8% | 1,500 |
| Search   | Conversion | 8% | 9.5% | 800 |

Let's compute the power for each campaign and decide which tests are reliable.

In [ ]:
campaigns = [
    {'name': 'Email (open rate)',      'p0': 0.20, 'p1': 0.23, 'n': 1000},
    {'name': 'Social (click rate)',    'p0': 0.03, 'p1': 0.038,'n': 1500},
    {'name': 'Search (conversion)',    'p0': 0.08, 'p1': 0.095,'n': 800},
]

alpha = 0.05
z_alpha_two = norm.ppf(1 - alpha / 2)   # two-tailed

results = []
for c in campaigns:
    pp = (c['p0'] + c['p1']) / 2
    se = np.sqrt(2 * pp * (1 - pp) / c['n'])
    z_b = (c['p1'] - c['p0']) / se - z_alpha_two
    pwr = norm.cdf(z_b)
    d   = (c['p1'] - c['p0']) / np.sqrt(pp * (1 - pp))  # Cohen's h approx
    results.append({'Campaign': c['name'],
                    'Control': f"{c['p0']:.1%}",
                    'Treatment': f"{c['p1']:.1%}",
                    'n/group': c['n'],
                    'Effect (d)': f"{d:.3f}",
                    'Power': f"{pwr:.1%}",
                    'Reliable?': '✅ Yes' if pwr >= 0.80 else '❌ Underpowered'})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Bar chart of powers
pwr_vals = [float(r['Power'].strip('%')) / 100 for r in results]
names    = [r['Campaign'] for r in results]
colors_c = ['seagreen' if p >= 0.80 else 'tomato' for p in pwr_vals]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, pwr_vals, color=colors_c, edgecolor='white')
ax.axvline(0.80, color='black', linestyle='--', linewidth=1.8, label='80% power target')
for bar, pwr in zip(bars, pwr_vals):
    ax.text(pwr + 0.01, bar.get_y() + bar.get_height()/2,
            f'{pwr:.0%}', va='center', fontsize=12, fontweight='bold')
ax.set_xlabel('Statistical Power', fontsize=12)
ax.set_title('Power of three simultaneous marketing A/B tests (α=0.05)', fontsize=13)
ax.set_xlim(0, 1.15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Reading the results

- **Email campaign:** High power → if the lift is real, we will almost certainly detect it. Results are trustworthy.
- **Social campaign:** Detecting a 0.8pp lift in a 3% base rate is very hard with 1,500 users — likely underpowered. The company needs more traffic.
- **Search campaign:** Check whether it meets the 80% threshold. If not, results should be interpreted cautiously.

> **Business rule:** Never interpret a "non-significant" result from an underpowered test as evidence that the campaign doesn't work. It may just be that **you didn't collect enough data**.

---

## 5. The p-value — What It Really Means

The **p-value** answers this question:

> *"If the null hypothesis were true, how likely is it to observe data as extreme as (or more extreme than) what I observed?"*

### Common misconceptions

| ❌ Wrong interpretation | ✅ Correct interpretation |
|------------------------|---------------------------|
| p-value = probability that $H_0$ is true | p-value = probability of this data **given** $H_0$ is true |
| Small p-value = large effect | Small p-value = strong evidence against $H_0$ (effect size is separate) |
| p > 0.05 means no effect | p > 0.05 means not enough evidence; the effect may still exist |
| p < 0.05 means the result matters | Statistical significance ≠ practical importance |

### Visual intuition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(-4, 4, 400)
y = norm.pdf(x)

for ax, observed, title in zip(axes,
                                [1.2, 2.5],
                                ['p = 0.23  (not significant at α=0.05)',
                                 'p = 0.01  (significant at α=0.05)']):
    p_shown = 2 * (1 - norm.cdf(abs(observed)))   # two-tailed
    ax.plot(x, y, 'steelblue', linewidth=2)
    ax.fill_between(x, y, where=(x >=  abs(observed)), alpha=0.45, color='tomato', label=f'p-value area ({p_shown:.3f}/2)')
    ax.fill_between(x, y, where=(x <= -abs(observed)), alpha=0.45, color='tomato')
    ax.axvline( observed, color='darkred', linestyle='--', linewidth=1.8, label=f'Observed statistic = {observed}')
    ax.axvline(-observed, color='darkred', linestyle='--', linewidth=1.8)
    ax.axvline( 1.96, color='gray', linestyle=':', linewidth=1.3, label='Critical value ±1.96 (α=0.05)')
    ax.axvline(-1.96, color='gray', linestyle=':', linewidth=1.3)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Test statistic')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('p-value: area in both tails beyond the observed statistic (two-tailed)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Type I and Type II Errors

Any decision from a hypothesis test can be right or wrong. The two possible errors are:

|  | $H_0$ is actually **True** | $H_0$ is actually **False** |
|--|--|--|
| **Reject $H_0$** | ❌ **Type I Error** ($\alpha$) | ✅ Correct (Power = $1 - \beta$) |
| **Fail to reject $H_0$** | ✅ Correct | ❌ **Type II Error** ($\beta$) |

### Definitions

- **Type I Error (False Positive):** We reject $H_0$ when it is actually true.
  - Controlled by the significance level $\alpha$.
  - *Real-world analogy:* A fire alarm goes off, but there is no fire.

- **Type II Error (False Negative):** We fail to reject $H_0$ when it is actually false.
  - Probability = $\beta$. We want to minimize $\beta$ (maximize **power** = $1 - \beta$).
  - *Real-world analogy:* There IS a fire, but the alarm never goes off.

### The trade-off

Lowering $\alpha$ reduces Type I errors but **increases** Type II errors (for a fixed sample size). The only way to reduce both simultaneously is to **increase the sample size**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.linspace(-4, 8, 600)
y0 = norm.pdf(x, loc=0, scale=1)
y1 = norm.pdf(x, loc=3, scale=1)
critical = norm.ppf(0.95)

ax.fill_between(x, y0, where=(x >= critical), alpha=0.5, color='tomato',   label='Type I Error (α = 0.05)')
ax.fill_between(x, y1, where=(x <= critical), alpha=0.5, color='steelblue',label='Type II Error (β)')
ax.plot(x, y0, 'darkred', linewidth=2, label='H₀ distribution (μ=0)')
ax.plot(x, y1, 'navy',    linewidth=2, label='H₁ distribution (μ=3)')
ax.axvline(critical, color='black', linestyle='--', linewidth=1.5, label=f'Critical value = {critical:.2f}')

beta_val  = norm.cdf(critical, loc=3, scale=1)
power_val = 1 - beta_val
ax.fill_between(x, y1, where=(x >= critical), alpha=0.4, color='seagreen', label=f'Power = {power_val:.0%}')

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Type I Error, Type II Error, and Power', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Critical value : {critical:.4f}')
print(f'α (Type I)     : 0.05')
print(f'β (Type II)    : {beta_val:.4f}')
print(f'Power (1 − β)  : {power_val:.4f}')

---

## 7. Sample Size Calculation

Before collecting data, we need to know: **how many observations do I need?**

For a **one-sample z-test**, the formula is:

$$
n = \left( \frac{(z_{\alpha/2} + z_{\beta}) \cdot \sigma}{\delta} \right)^2
$$

For a **two-sample t-test** (comparing two independent groups):

$$
n_{\text{per group}} = 2 \left( \frac{(z_{\alpha/2} + z_{\beta}) \cdot \sigma}{\delta} \right)^2
$$

Where:
- $z_{\alpha/2}$ = critical value for the significance level (e.g., 1.96 for $\alpha=0.05$, two-tailed)
- $z_{\beta}$ = critical value for desired power (e.g., 0.84 for 80% power)
- $\sigma$ = estimated population standard deviation
- $\delta$ = minimum detectable effect

In [ ]:
def sample_size_one_sample(effect_size, alpha=0.05, power=0.80):
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    return int(np.ceil(((z_alpha + z_beta) / effect_size) ** 2))

def sample_size_two_sample(effect_size, alpha=0.05, power=0.80):
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    return int(np.ceil(2 * ((z_alpha + z_beta) / effect_size) ** 2))

effect_sizes = np.linspace(0.1, 1.0, 50)
powers_ss = [0.70, 0.80, 0.90]
colors_ss = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(10, 6))
for pwr, color in zip(powers_ss, colors_ss):
    ns = [sample_size_two_sample(e, power=pwr) for e in effect_sizes]
    ax.plot(effect_sizes, ns, linewidth=2, color=color, label=f'Power = {int(pwr*100)}%')

ax.set_xlabel("Effect size (Cohen's d = δ/σ)", fontsize=12)
ax.set_ylabel('Sample size per group', fontsize=12)
ax.set_title('Required sample size vs. effect size (α = 0.05, two-sample test)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 800)
plt.tight_layout()
plt.show()

print("Quick reference — two-sample test (α=0.05, power=80%)")
print(f"{'Effect size (d)':<20} {'n per group':<15}")
print("-" * 35)
for d_label, d_val in [('Small (0.2)', 0.2), ('Medium (0.5)', 0.5), ('Large (0.8)', 0.8)]:
    n = sample_size_two_sample(d_val)
    print(f"{d_label:<20} {n:<15}")

---

## 8. Concrete Example 1 — One-Sample t-test

### Scenario

A soft-drink company claims their bottles contain **500 mL** on average. A quality inspector measures **30 randomly selected bottles**.

- $H_0: \mu = 500$ mL
- $H_1: \mu \neq 500$ mL
- $\alpha = 0.05$

In [ ]:
mu_claimed = 500
true_mu    = 497
sigma      = 8
n          = 30

sample = np.random.normal(loc=true_mu, scale=sigma, size=n)
print(f"Sample mean  : {sample.mean():.2f} mL")
print(f"Sample std   : {sample.std(ddof=1):.2f} mL")

t_stat, p_value = ttest_1samp(sample, popmean=mu_claimed)
print(f"\nt-statistic  : {t_stat:.4f}")
print(f"p-value      : {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nDecision: Reject H₀  →  Evidence of mis-calibration.")
else:
    print(f"\nDecision: Fail to reject H₀  →  No significant deviation found.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
alpha = 0.05
df = n - 1

axes[0].hist(sample, bins=10, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(sample.mean(), color='darkred',  linewidth=2, linestyle='--', label=f'Sample mean = {sample.mean():.1f}')
axes[0].axvline(mu_claimed,    color='seagreen', linewidth=2, linestyle=':',  label=f'Claimed mean = {mu_claimed}')
axes[0].set_xlabel('Volume (mL)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Sample distribution of bottle volumes', fontsize=13)
axes[0].legend()

x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, df=df)
crit = stats.t.ppf(1 - alpha / 2, df=df)

axes[1].plot(x, y, 'navy', linewidth=2)
axes[1].fill_between(x, y, where=(x >=  crit), color='tomato', alpha=0.5, label='Rejection region')
axes[1].fill_between(x, y, where=(x <= -crit), color='tomato', alpha=0.5)
axes[1].axvline(t_stat, color='darkred', linewidth=2, linestyle='--', label=f't-stat = {t_stat:.2f}')
axes[1].set_xlabel('t value', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title(f't-distribution (df={df})  —  p = {p_value:.4f}', fontsize=13)
axes[1].legend()
plt.tight_layout()
plt.show()

---

## 9. Concrete Example 2 — Two-Sample t-test (A/B Test)

### Scenario

An e-commerce company tests a new **simplified checkout page** against the current one, measuring time to purchase (seconds).

- $H_0: \mu_A = \mu_B$
- $H_1: \mu_A > \mu_B$ (new page is faster)
- $\alpha = 0.05$

In [ ]:
n_A, n_B = 80, 80
group_A = np.random.normal(loc=120, scale=20, size=n_A)
group_B = np.random.normal(loc=110, scale=18, size=n_B)

print("Group A (Control)   — mean: {:.1f}s,  std: {:.1f}s".format(group_A.mean(), group_A.std(ddof=1)))
print("Group B (Treatment) — mean: {:.1f}s,  std: {:.1f}s".format(group_B.mean(), group_B.std(ddof=1)))

t_stat, p_two = ttest_ind(group_A, group_B, equal_var=False)
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"\nWelch's t-statistic      : {t_stat:.4f}")
print(f"p-value (one-tailed A>B) : {p_one:.4f}")

pooled_std = np.sqrt((group_A.var(ddof=1) + group_B.var(ddof=1)) / 2)
cohens_d = (group_A.mean() - group_B.mean()) / pooled_std
print(f"Cohen's d (effect size)  : {cohens_d:.4f}")

if p_one < 0.05:
    print("\nDecision: Reject H₀ — the new checkout page is significantly faster.")
else:
    print("\nDecision: Fail to reject H₀.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([group_A, group_B], labels=['Group A\n(Control)', 'Group B\n(Treatment)'],
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='darkred', linewidth=2))
axes[0].set_ylabel('Checkout time (seconds)', fontsize=12)
axes[0].set_title('Distribution of checkout times', fontsize=13)

axes[1].hist(group_A, bins=15, alpha=0.6, color='steelblue',  label='Group A (Control)',   density=True)
axes[1].hist(group_B, bins=15, alpha=0.6, color='darkorange', label='Group B (Treatment)', density=True)
axes[1].axvline(group_A.mean(), color='steelblue',  linestyle='--', linewidth=2)
axes[1].axvline(group_B.mean(), color='darkorange', linestyle='--', linewidth=2)
axes[1].set_xlabel('Checkout time (seconds)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Overlapping distributions', fontsize=13)
axes[1].legend()

plt.suptitle('A/B Test: New vs Current Checkout Page', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 10. Planning the A/B Test — Sample Size

Before running the test, the product team sets:
- $\sigma \approx 20$ seconds (historical), $\delta = 10$ seconds (minimum useful lift)
- $\alpha = 0.05$, power = 80%

In [ ]:
sigma = 20
delta = 10
effect_size = delta / sigma
print(f"Effect size (Cohen's d) = {delta}/{sigma} = {effect_size:.2f}\n")

for alpha_val, power_val in [(0.05, 0.80), (0.05, 0.90), (0.01, 0.80), (0.01, 0.90)]:
    n = sample_size_two_sample(effect_size, alpha=alpha_val, power=power_val)
    print(f"α={alpha_val}  power={int(power_val*100)}%  →  n = {n} per group  (total = {2*n} users)")

---

## 11. Type I & II Errors — Medical Testing

| Error | Event | Real-world consequence |
|-------|-------|------------------------|
| **Type I** | Test says "disease" — patient is healthy | Unnecessary treatment, cost, anxiety |
| **Type II** | Test says "healthy" — patient is sick | Disease untreated — potentially fatal |

In [ ]:
np.random.seed(99)
n_patients = 10_000
has_disease  = np.random.binomial(1, 0.10, n_patients).astype(bool)

sensitivity = 0.85
specificity = 0.92

test_positive = np.where(
    has_disease,
    np.random.binomial(1, sensitivity,       n_patients),
    np.random.binomial(1, 1 - specificity,   n_patients)
).astype(bool)

TP = ( has_disease &  test_positive).sum()
TN = (~has_disease & ~test_positive).sum()
FP = (~has_disease &  test_positive).sum()  # Type I
FN = ( has_disease & ~test_positive).sum()  # Type II

print("Confusion Matrix (10,000 patients)")
print(f"{'':25} {'Test +':>10} {'Test -':>10}")
print(f"{'Disease (actual +)':25} {TP:>10} {FN:>10}  ← FN = Type II")
print(f"{'No disease (actual -)':25} {FP:>10} {TN:>10}  ← FP = Type I")
print(f"\nType I  errors : {FP}  ({FP/n_patients*100:.1f}%)")
print(f"Type II errors : {FN}  ({FN/n_patients*100:.1f}%)")
print(f"Power          : {TP/(TP+FN)*100:.1f}%")

---

## 12. Summary & Key Takeaways

| Concept | One-liner |
|---------|----------|
| Hypothesis test | Formal procedure to decide if data provides enough evidence against $H_0$ |
| $\alpha$ (significance) | Max tolerated false alarm rate. Set **before** collecting data. |
| Power ($1-\beta$) | Probability of detecting a real effect. Target ≥ 80%. |
| p-value | Probability of the observed data (or more extreme) **given** $H_0$ is true |
| Type I error | Reject $H_0$ when it's true — false alarm, controlled by $\alpha$ |
| Type II error | Fail to reject $H_0$ when it's false — missed detection, controlled by $\beta$ |
| Effect size | How big is the difference? Statistical significance ≠ practical importance |
| Sample size | Larger $n$ → lower $\beta$ → higher power (for fixed $\alpha$ and effect size) |

### The workflow

```
1. Define H₀ and H₁ clearly.
2. Choose α based on the cost of a false alarm.
3. Choose target power based on the cost of a missed detection.
4. Estimate σ and δ (minimum effect that matters in practice).
5. Calculate required sample size.
6. Collect data.
7. Compute the test statistic and p-value.
8. Compare p-value to α and decide.
9. Report effect size alongside p-value.
10. Translate into a real-world business conclusion.
```

---

## Practice Exercises

**Exercise 1 — Significance level**  
A startup is testing whether a new chatbot increases customer satisfaction scores above the current mean of 70/100. They set α = 0.10.  
(a) What does this α mean in plain English?  
(b) Would you recommend a stricter or more lenient α? Justify your answer.

**Exercise 2 — Power**  
Using the `compute_power` function defined above, calculate the power of a one-sample z-test when:  
- Effect size d = 0.25, n = 50, α = 0.05  
- Effect size d = 0.25, n = 200, α = 0.05  
What sample size is needed to reach 80% power?

**Exercise 3 — One-sample t-test**  
A bag of chips claims to weigh 150 g. You sample 25 bags and find mean = 147 g, std = 6 g. At α = 0.05, is there evidence the bags are under-filled?

**Exercise 4 — Two-sample t-test**  
Group A students study 4 hours/day (std = 1.2, n = 35). Group B (new app) averages 4.6 hours (std = 1.1, n = 35). Does the app significantly increase study time? (α = 0.05)

**Exercise 5 — Campaign power**  
A company runs an email campaign A/B test. Current click rate = 12%. They expect the new design to achieve 14%. With 600 users per group and α = 0.05 (two-tailed):  
(a) Calculate the power of this test.  
(b) How many users per group do they need to reach 80% power?

In [ ]:
# Exercise 1 — Answer in comments or markdown


In [ ]:
# Exercise 2 — Your code here


In [ ]:
# Exercise 3 — Your code here


In [ ]:
# Exercise 4 — Your code here


In [ ]:
# Exercise 5 — Your code here
